<a href="https://colab.research.google.com/github/crystalloide/IA_102_IA_Agentique/blob/main/IA102_0_Pr%C3%A9sentation_vue_d_ensemble_%C3%A9cosyst%C3%A8me_LangGraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Aperçu de LangGraph** :

Utilisé par des entreprises majeures (dont Klarna, Uber, JP Morgan et bien d'autres), LangGraph est un framework d'orchestration et un environnement d'exécution de bas niveau permettant de créer, gérer et déployer des agents persistants et à état.

LangGraph offre un contrôle précis permettant de combiner des étapes déterministes codées manuellement avec des étapes pilotées par LLM au sein d'un même graphe.

Il est possible ainsi de créer des agents sur mesure qui se comportent exactement comme une application l'exige.

LangGraph est un système de bas niveau, entièrement dédié à l'orchestration d'agents.

Avant d'utiliser LangGraph, il est recommandé de se familiariser avec certains composants nécessaires à la création d'agents, en commençant par les modèles et les outils.

Nous utiliserons fréquemment les composants LangChain dans la documentation pour intégrer les modèles et les outils, mais LangGraph n'en a pas besoin.

Si vous débutez avec les agents ou si vous souhaitez une abstraction de plus haut niveau, nous vous recommandons d'utiliser les agents LangChain qui proposent des architectures préconfigurées pour les boucles LLM et d'appel d'outils courantes.

LangGraph se concentre sur les capacités sous-jacentes importantes pour l'orchestration des agents : exécution durable, streaming, intervention humaine et plus encore.

L'un des principaux atouts de LangGraph réside dans sa capacité à combiner des étapes déterministes et des étapes agentiques pilotées par LLM au sein d'un même graphe.

Ceci permet de créer des flux de travail sur mesure où certaines parties de la logique sont entièrement prévisibles et auditables, tandis que d'autres sont flexibles et pilotées par un modèle, offrant ainsi un contrôle précis sur l'application de l'IA.

- **Deep Agents** est un ensemble d'agents : planification, sous-agents, outils de système de fichiers et gestion du contexte sur LangGraph.

- **LangChain** est le framework d'agents : abstractions et intégrations pour les modèles, les outils et les boucles d'agents.

- **LangGraph** est l'environnement d'exécution d'orchestration : exécution durable, flux continu, intervention humaine et persistance.

- **LangSmith** est la plateforme de traçage, d'évaluation, d'invites et de déploiement pour différents frameworks.

- **LangSmith Engine** détecte les problèmes dans les traces de votre agent LangGraph et propose des correctifs. Vous pouvez soumettre une demande d'extraction avec le correctif proposé directement depuis l'onglet Engine.

- **LangSmith Fleet** est le générateur d'agents sans code pour les modèles, les intégrations et l'automatisation des tâches routinières.

## Déroulé du handlab

| Partie | Sujet | Ce que l'on manipule |
|---|---|---|
| 0 | Préparation | installation figée des versions, fichiers de test, utilitaires |
| 1 | **LangChain** | modèles, invites, outils, boucle d'agent `create_agent`, middleware, sortie structurée |
| 2 | **LangGraph** | étapes déterministes + agentiques, persistance, exécution durable, flux continu (streaming), intervention humaine, mémoire |
| 3 | **Deep Agents** | planification (`write_todos`), sous-agents (`task`), système de fichiers, gestion du contexte |
| 4 | **LangSmith** | traçage, évaluation, invites versionnées, déploiement sur un Agent Server local |
| 5 | **LangSmith Engine** | détection de problèmes dans les traces, diagnostic, correctif, validation (simulation locale du cycle) |
| 6 | **LangSmith Fleet** | agent défini sans code (spécification), intégrations, approbations, déclencheur, appel depuis le code |

> **Aucune clé API n'est nécessaire.** Toutes les cellules utilisent un *modèle simulé* déterministe, ce qui garantit
> que le notebook s'exécute de bout en bout (menu *Exécution → Tout exécuter*) avec des résultats reproductibles.
> Les cellules marquées **(optionnel)** utilisent un vrai modèle ou le vrai service LangSmith **uniquement si** la
> variable d'environnement correspondante est définie ; sinon elles s'ignorent proprement.

# Partie 0 — Préparation de l'environnement

## 0.1 Installation (versions figées)

Les versions sont **figées** pour garantir la compatibilité entre bibliothèques (versions stables publiées au 24 septembre 2026) :

| Paquet | Version | Rôle |
|---|---|---|
| `langgraph` | 1.2.12 | environnement d'exécution d'orchestration |
| `langchain` / `langchain-core` | 1.4.2 / 1.6.5 | framework d'agents |
| `deepagents` | 0.7.19 | agents « profonds » construits sur LangGraph |
| `langsmith` | 0.14.0 | SDK de la plateforme LangSmith |
| `langgraph-checkpoint-sqlite` | 3.1.1 | persistance SQLite |

> **Compatibilité Colab** : `deepagents` déclare une dépendance vers `langchain-google-genai`, qui exige `google-auth ≥ 2.56`
> alors que Colab verrouille `google-auth==2.49.0` (paquet `google-colab`). Comme ce handlab n'utilise pas les modèles Gemini,
> `deepagents` est installé avec `--no-deps` et ses autres dépendances sont installées explicitement : aucun paquet système de Colab
> n'est modifié et aucun redémarrage de session n'est nécessaire. La cellule suivante le vérifie.

In [1]:
# 1) Bibliothèques de l'écosystème (versions figées, sans toucher aux paquets système de Colab)
%pip install -q "langgraph==1.2.12" "langgraph-checkpoint==4.2.0" "langgraph-prebuilt==1.1.0" "langgraph-sdk==0.4.5" "langgraph-checkpoint-sqlite==3.1.1" "langchain==1.4.2" "langchain-core==1.6.5" "langchain-anthropic==1.7.4" "langsmith==0.14.0" "wcmatch==11.0.1"
# 2) deepagents sans ses dépendances optionnelles Google (qui imposeraient une mise à jour de google-auth, verrouillé par Colab)
%pip install -q --no-deps "deepagents==0.7.19"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.2/250.2 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.3/162.3 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.7/163.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 805.0/805.0 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.4/163.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.8/337.8 kB 2.2 MB/s eta 0:00:00


In [2]:
# Vérification des versions réellement chargées
import sys, importlib.metadata as md

print("Python", sys.version.split()[0])
for paquet in ["langgraph", "langgraph-checkpoint", "langgraph-checkpoint-sqlite", "langgraph-sdk",
               "langchain", "langchain-core", "deepagents", "langsmith"]:
    print(f"  {paquet:<28} {md.version(paquet)}")

assert sys.version_info >= (3, 11), "deepagents nécessite Python ≥ 3.11"
assert md.version("langgraph").startswith("1."), "LangGraph 1.x attendu"
try:
    print(f"  {'google-auth (Colab)':<28} {md.version('google-auth')}  ← inchangé")
except md.PackageNotFoundError:
    pass
import deepagents  # noqa: F401  (vérifie que deepagents s'importe sans langchain-google-genai)
print("\n✅ Environnement prêt")

Python 3.13.15
  langgraph                    1.2.12
  langgraph-checkpoint         4.2.0
  langgraph-checkpoint-sqlite  3.1.1
  langgraph-sdk                0.4.5
  langchain                    1.4.2
  langchain-core               1.6.5
  deepagents                   0.7.19
  langsmith                    0.14.0
  google-auth (Colab)          2.49.0  ← inchangé

✅ Environnement prêt


## 0.2 Génération des fichiers de test

Cette cellule crée **tous** les fichiers utilisés par le handlab dans le dossier `lab_ecosysteme/` (elle peut être ré-exécutée sans risque : le dossier est recréé) :

- `data/catalogue.csv` : un mini-catalogue de formations (outils LangChain) ;
- `data/jeu_evaluation.jsonl` : un jeu d'évaluation (LangSmith) ;
- `data/tickets.csv` : des tickets de support (LangSmith Fleet) ;
- `espace_agent/` : l'espace de travail du Deep Agent (documents + `AGENTS.md`) ;
- `engine_demo/outils_support.py` : un outil **volontairement défectueux** (LangSmith Engine) ;
- `fleet/agent_spec.json` : la spécification « sans code » d'un agent (LangSmith Fleet) ;
- `projet_deploiement/` : un projet d'agent prêt à déployer (`langgraph.json`, `agent.py`).

In [3]:
import json, shutil, textwrap
from pathlib import Path

LAB = Path("lab_ecosysteme").resolve()
if LAB.exists():
    shutil.rmtree(LAB)
for sous_dossier in ["data", "espace_agent/docs", "engine_demo", "fleet/memoires", "projet_deploiement"]:
    (LAB / sous_dossier).mkdir(parents=True, exist_ok=True)

def ecrire(chemin, contenu):
    chemin = LAB / chemin
    chemin.write_text(textwrap.dedent(contenu).lstrip(), encoding="utf-8")
    return chemin

# --- Catalogue de formations -------------------------------------------------
ecrire("data/catalogue.csv", """
code,titre,duree_jours,niveau
AIR-101,Apache Airflow - fondamentaux,2,debutant
AIR-201,Apache Airflow - orchestration avancée,3,avance
KAF-101,Apache Kafka - fondamentaux,2,debutant
SPK-201,Apache Spark - optimisation,3,avance
LGR-101,LangGraph - agents avec état,2,intermediaire
LGR-201,LangGraph - déploiement et observabilité,1,avance
K8S-101,Kubernetes - fondamentaux,3,debutant
""")

# --- Jeu d'évaluation ----------------------------------------------------------
exemples = [
    {"question": "Quels modules sur airflow ? Durée totale ?", "duree_attendue": 5, "codes_attendus": ["AIR-101", "AIR-201"]},
    {"question": "Quels modules sur langgraph ? Durée totale ?", "duree_attendue": 3, "codes_attendus": ["LGR-101", "LGR-201"]},
    {"question": "Quels modules sur kafka ? Durée totale ?", "duree_attendue": 2, "codes_attendus": ["KAF-101"]},
    {"question": "Quels modules sur kubernetes ? Durée totale ?", "duree_attendue": 3, "codes_attendus": ["K8S-101"]},
]
(LAB / "data/jeu_evaluation.jsonl").write_text(
    "\n".join(json.dumps(e, ensure_ascii=False) for e in exemples), encoding="utf-8")

# --- Tickets de support (Fleet) ---------------------------------------------
ecrire("data/tickets.csv", """
id,client,sujet,priorite
T-001,ACME,Le lab Kubernetes ne démarre pas,haute
T-002,Globex,Demande de facture,basse
T-003,Initech,Notebook Airflow en erreur,haute
T-004,Umbrella,Question sur le planning,moyenne
""")

# --- Espace de travail du Deep Agent ------------------------------------------
ecrire("espace_agent/docs/langgraph.md", """
# LangGraph
Environnement d'exécution d'orchestration : exécution durable, flux continu, intervention humaine et persistance.
""")
ecrire("espace_agent/docs/langchain.md", """
# LangChain
Framework d'agents : abstractions et intégrations pour les modèles, les outils et les boucles d'agents.
""")
ecrire("espace_agent/docs/langsmith.md", """
# LangSmith
Plateforme de traçage, d'évaluation, d'invites et de déploiement, indépendante du framework.
""")
ecrire("espace_agent/AGENTS.md", """
# Consignes permanentes de l'agent
- Toujours répondre en français.
- Écrire les rapports dans le dossier /rapports/.
""")

# --- Outil volontairement défectueux (LangSmith Engine) -----------------------
ecrire("engine_demo/outils_support.py", """
import json

STOCK = {"SKU-1": 12, "SKU-2": 0, "SKU-3": 5}


def consulter_stock(reference: str) -> str:
    \"\"\"Renvoie le stock d'une référence produit au format JSON.\"\"\"
    try:
        ref = reference.strip().upper()
        return json.dumps({"reference": ref, "stock": STOCK[ref]})
    except Exception as exc:
        # BUG : l'erreur est « avalée » et renvoyée comme un texte ordinaire
        return f"erreur: référence inconnue ({exc})"
""")

# --- Spécification « sans code » d'un agent Fleet -----------------------------
spec_fleet = {
    "nom": "Assistant de tri des tickets",
    "instructions": "Chaque matin, lis les tickets, rédige un rapport de tri et préviens l'équipe support par email.",
    "integrations": ["tickets", "email"],
    "approbations": {"envoyer_email": True},
    "declencheur": {"type": "planification", "cron": "0 8 * * 1-5", "fuseau": "Europe/Paris"},
    "memoire": "/memoires/AGENTS.md",
}
(LAB / "fleet/agent_spec.json").write_text(json.dumps(spec_fleet, ensure_ascii=False, indent=2), encoding="utf-8")
ecrire("fleet/memoires/AGENTS.md", """
# Mémoire de l'agent de tri
- L'équipe support se joint à l'adresse support@exemple.fr.
- Les tickets de priorité « haute » doivent apparaître en premier.
""")

# --- Projet prêt à déployer (Agent Server) ------------------------------------
ecrire("projet_deploiement/agent.py", """
import csv
from pathlib import Path
from langgraph.graph import StateGraph, MessagesState, START, END

CATALOGUE = Path(__file__).parent / "catalogue.csv"


def repondre(state: MessagesState):
    question = state["messages"][-1].content.lower()
    with open(CATALOGUE, encoding="utf-8") as f:
        modules = [m for m in csv.DictReader(f) if any(mot in m["titre"].lower() for mot in question.split())]
    if not modules:
        return {"messages": [{"role": "ai", "content": "Aucun module trouvé."}]}
    liste = ", ".join(f"{m['code']} ({m['duree_jours']} j)" for m in modules)
    return {"messages": [{"role": "ai", "content": f"Modules trouvés : {liste}"}]}


builder = StateGraph(MessagesState)
builder.add_node(repondre)
builder.add_edge(START, "repondre")
builder.add_edge("repondre", END)
graph = builder.compile()
""")
shutil.copy(LAB / "data/catalogue.csv", LAB / "projet_deploiement/catalogue.csv")
(LAB / "projet_deploiement/langgraph.json").write_text(json.dumps(
    {"dependencies": ["."], "graphs": {"agent_catalogue": "./agent.py:graph"}}, indent=2), encoding="utf-8")
ecrire("projet_deploiement/requirements.txt", """
langgraph==1.2.12
""")

for f in sorted(LAB.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(LAB)}  ({f.stat().st_size} o)")
print("\n✅ Fichiers de test générés dans", LAB)

  data/catalogue.csv  (387 o)
  data/jeu_evaluation.jsonl  (472 o)
  data/tickets.csv  (209 o)
  engine_demo/outils_support.py  (455 o)
  espace_agent/AGENTS.md  (121 o)
  espace_agent/docs/langchain.md  (118 o)
  espace_agent/docs/langgraph.md  (128 o)
  espace_agent/docs/langsmith.md  (109 o)
  fleet/agent_spec.json  (411 o)
  fleet/memoires/AGENTS.md  (162 o)
  projet_deploiement/agent.py  (831 o)
  projet_deploiement/catalogue.csv  (387 o)
  projet_deploiement/langgraph.json  (96 o)
  projet_deploiement/requirements.txt  (18 o)

✅ Fichiers de test générés dans /content/lab_ecosysteme


## 0.3 Utilitaires communs : un modèle simulé et un traceur local

- **`ModeleSimule`** est un vrai `BaseChatModel` LangChain : il s'utilise exactement comme `ChatOpenAI` ou `ChatAnthropic`
  (`invoke`, `bind_tools`, compatibilité avec `create_agent`, LangGraph, Deep Agents…). Sa réponse est produite par une
  **politique** Python déterministe (au lieu d'un LLM), ce qui rend le handlab reproductible et gratuit.
- **`TraceurLocal`** est un *tracer* LangChain qui conserve l'arbre des exécutions (runs) en mémoire : c'est la même
  structure de données que celle envoyée à LangSmith.

In [4]:
import json, re, time, uuid, logging, warnings
from typing import Callable, List
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.tracers.base import BaseTracer

warnings.filterwarnings("ignore", category=UserWarning, module="langsmith")
logging.getLogger("langsmith").setLevel(logging.CRITICAL)


class ModeleSimule(BaseChatModel):
    """Chat model déterministe : la réponse est calculée par une fonction `politique(messages) -> AIMessage`."""
    politique: Callable[[List[BaseMessage]], AIMessage]

    @property
    def _llm_type(self) -> str:
        return "modele-simule"

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        reponse = self.politique(messages)
        entree = sum(len(str(m.content).split()) for m in messages)
        sortie = len(str(reponse.content).split()) + 5 * len(reponse.tool_calls)
        reponse.usage_metadata = {"input_tokens": entree, "output_tokens": sortie, "total_tokens": entree + sortie}
        return ChatResult(generations=[ChatGeneration(message=reponse)])

    def bind_tools(self, tools, **kwargs):
        # Un vrai modèle recevrait ici le schéma JSON des outils ; notre politique les connaît déjà.
        return self


def appel(nom: str, **arguments) -> dict:
    """Fabrique un appel d'outil (tool call) tel qu'un LLM le produirait."""
    return {"name": nom, "args": arguments, "id": "call_" + uuid.uuid4().hex[:8], "type": "tool_call"}


def messages_outils(messages):
    """Résultats d'outils (ToolMessage) reçus depuis la dernière question de l'utilisateur."""
    resultats = []
    for m in reversed(messages):
        if isinstance(m, HumanMessage):
            break
        if isinstance(m, ToolMessage):
            resultats.insert(0, m)
    return resultats


def derniere_question(messages) -> str:
    return next(m.content for m in reversed(messages) if isinstance(m, HumanMessage))


def afficher_conversation(messages, largeur=220):
    for m in messages:
        role = type(m).__name__.replace("Message", "")
        texte = str(m.content).replace("\n", " ")
        texte = texte if len(texte) <= largeur else texte[:largeur] + "…"
        if getattr(m, "tool_calls", None):
            appels = ", ".join(f"{t['name']}({json.dumps(t['args'], ensure_ascii=False)[:90]})" for t in m.tool_calls)
            texte = (texte + " " if texte else "") + f"🔧 {appels}"
        nom = f"[{m.name}]" if isinstance(m, ToolMessage) else ""
        print(f"{role:>6}{nom}: {texte}")


class TraceurLocal(BaseTracer):
    """Conserve en mémoire les arbres d'exécution (même structure que les traces LangSmith)."""
    def __init__(self):
        super().__init__()
        self.traces = []

    def _persist_run(self, run):
        self.traces.append(run)


def afficher_arbre(run, profondeur=0):
    duree = (run.end_time - run.start_time).total_seconds() * 1000 if run.end_time else 0
    jetons = ""
    if run.run_type == "llm":
        try:
            usage = run.outputs["generations"][0][0]["message"].usage_metadata
            jetons = f"  jetons={usage['total_tokens']}"
        except Exception:
            pass
    statut = "❌" if run.error else "✓"
    print(f"{'   ' * profondeur}{statut} [{run.run_type}] {run.name}  {duree:.1f} ms{jetons}")
    for enfant in sorted(run.child_runs, key=lambda r: r.dotted_order or ""):
        afficher_arbre(enfant, profondeur + 1)


print("✅ Utilitaires prêts : ModeleSimule, appel, TraceurLocal, afficher_conversation, afficher_arbre")

✅ Utilitaires prêts : ModeleSimule, appel, TraceurLocal, afficher_conversation, afficher_arbre


## 0.4 Premier graphe LangGraph (exemple de la documentation)

https://docs.langchain.com/oss/python/langgraph/overview

In [5]:
from langgraph.graph import StateGraph, MessagesState, START, END

def mock_llm(state: MessagesState):
    return {"messages": [{"role": "ai", "content": "hello world"}]}

graph = StateGraph(MessagesState)
graph.add_node(mock_llm)
graph.add_edge(START, "mock_llm")
graph.add_edge("mock_llm", END)
graph = graph.compile()

graph.invoke({"messages": [{"role": "user", "content": "hi!"}]})

{'messages': [HumanMessage(content='hi!', additional_kwargs={}, response_metadata={}, id='2461b985-85c3-4ddd-83f9-5b2c35b09f1b'),
  AIMessage(content='hello world', additional_kwargs={}, response_metadata={}, id='cb8f5745-04cc-418e-8b0d-1d1ccf279645', tool_calls=[], invalid_tool_calls=[])]}

LangGraph fournit une infrastructure de bas niveau pour tout flux de travail ou agent de longue durée et avec état.

## Bénéfices principaux :

LangGraph n'abstrait ni les invites ni l'architecture.

LangGraph offre les principaux avantages suivants :
- **Combinez étapes déterministes et agentiques** : associez une logique déterministe codée manuellement à une prise de décision pilotée par un modèle linéaire logique (LLM) au sein d’un même graphe. Utilisez les étapes déterministes lorsque la fiabilité et la prévisibilité sont essentielles, et les étapes agentiques là où la flexibilité est nécessaire, ce qui vous permet de contrôler précisément chaque aspect du comportement de votre agent.
- **Persistance** : Créez des agents qui persistent malgré les pannes et peuvent fonctionner pendant de longues périodes, en reprenant là où ils s'étaient arrêtés.
- **Intervention humaine** : Intégrez la supervision humaine en inspectant et en modifiant l'état de l'agent à tout moment.
- **Mémoire complète** : Créez des agents à état dotés à la fois d'une mémoire de travail à court terme pour le raisonnement continu et d'une mémoire à long terme sur plusieurs sessions.
- **Débogage avec LangSmith** : Bénéficiez d’une visibilité approfondie sur le comportement complexe des agents grâce à des outils de visualisation qui tracent les chemins d’exécution, capturent les transitions d’état et fournissent des métriques d’exécution détaillées.
- **Déploiement prêt pour la production** : Déployez en toute confiance des systèmes d’agents sophistiqués grâce à une infrastructure évolutive conçue pour gérer les défis uniques des flux de travail à état et de longue durée.
​
## Écosystème LangGraph :

Bien que LangGraph puisse être utilisé seul, il s'intègre également parfaitement à tous les produits LangChain, offrant ainsi aux développeurs une suite complète d'outils pour la création d'agents.

Pour optimiser le développement de votre application LLM, associez LangGraph à :

- **Observabilité avec LangSmith**
Centralisez le suivi des requêtes, l'évaluation des résultats et la surveillance des déploiements. Créez des prototypes en local avec LangGraph, puis déployez-les en production grâce à une observabilité et une évaluation intégrées afin de concevoir des systèmes d'agents plus fiables.
  https://docs.langchain.com/langsmith/observability

- **Déploiement avec LangSmith**
Déployez et faites évoluer vos agents en toute simplicité grâce à une plateforme de déploiement dédiée aux flux de travail continus et avec état. Découvrez, réutilisez, configurez et partagez des agents entre équipes et itérez rapidement grâce au prototypage visuel dans Studio.

  https://docs.langchain.com/langsmith/deployment

- **LangChain**
Fournit des intégrations et des composants composables pour simplifier le développement d'applications LLM. Contient des abstractions d'agents construites sur LangGraph.

  https://docs.langchain.com/oss/python/langchain/overview
​

____
**Remerciements**

LangGraph s'inspire de Pregel et d'Apache Beam . Son interface publique est inspirée de NetworkX . LangGraph est développé par LangChain Inc., les créateurs de LangChain, mais peut être utilisé indépendamment de LangChain.

# Partie 1 — LangChain : le framework d'agents

> **LangChain** est le framework d'agents : abstractions et intégrations pour les **modèles**, les **outils** et les **boucles d'agents**.

Depuis LangChain 1.0, l'API d'agent unique est `langchain.agents.create_agent`. Elle construit **un graphe LangGraph**
(boucle *modèle → outils → modèle*) et s'enrichit par des **middlewares**.

## 1.1 Modèles, messages et invites (prompts)

Tous les modèles partagent la même interface (`invoke`, `batch`, `stream`) et s'échangent des **messages** typés.
Les invites (`ChatPromptTemplate`) se composent avec les modèles grâce à l'opérateur `|`.

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


def politique_formateur(messages):
    consigne = next((m.content for m in messages if isinstance(m, SystemMessage)), "(aucune consigne)")
    return AIMessage(content=f"[{consigne}] Réponse simulée à : « {derniere_question(messages)} »")


modele = ModeleSimule(politique=politique_formateur)

# 1) Appel direct avec une liste de messages
reponse = modele.invoke([SystemMessage("Tu es formateur"), HumanMessage("Qu'est-ce qu'un agent ?")])
print("invoke  →", reponse.content)
print("usage   →", reponse.usage_metadata)

# 2) Invite paramétrée + modèle + analyseur de sortie = une chaîne (Runnable)
invite = ChatPromptTemplate.from_messages([
    ("system", "Tu es formateur en {domaine}"),
    ("human", "{question}"),
])
chaine = invite | modele | StrOutputParser()
print("chaîne  →", chaine.invoke({"domaine": "data engineering", "question": "Définis LangGraph en une phrase."}))

# 3) Traitement par lot
for r in chaine.batch([{"domaine": "Kubernetes", "question": "Qu'est-ce qu'un pod ?"},
                       {"domaine": "Kafka", "question": "Qu'est-ce qu'un topic ?"}]):
    print("batch   →", r)

invoke  → [Tu es formateur] Réponse simulée à : « Qu'est-ce qu'un agent ? »
usage   → {'input_tokens': 7, 'output_tokens': 13, 'total_tokens': 20}
chaîne  → [Tu es formateur en data engineering] Réponse simulée à : « Définis LangGraph en une phrase. »
batch   → [Tu es formateur en Kubernetes] Réponse simulée à : « Qu'est-ce qu'un pod ? »
batch   → [Tu es formateur en Kafka] Réponse simulée à : « Qu'est-ce qu'un topic ? »


## 1.2 Outils et boucle d'agent (`create_agent`)

Un **outil** est une fonction Python décorée par `@tool` : sa signature et sa docstring deviennent le schéma que le modèle reçoit.
La **boucle d'agent** alterne : le modèle décide d'appeler un outil → l'outil s'exécute → son résultat revient au modèle → … jusqu'à la réponse finale.

Ici, l'agent enchaîne **deux** appels d'outils sur le fichier `data/catalogue.csv`.

In [7]:
import csv
from langchain_core.tools import tool
from langchain.agents import create_agent

CATALOGUE = LAB / "data/catalogue.csv"


@tool
def chercher_module(mot_cle: str) -> str:
    """Cherche dans le catalogue les modules dont le titre contient le mot-clé. Renvoie une liste JSON."""
    with open(CATALOGUE, encoding="utf-8") as f:
        modules = [m for m in csv.DictReader(f) if mot_cle.lower() in m["titre"].lower()]
    return json.dumps(modules, ensure_ascii=False)


@tool
def duree_totale(codes: list[str]) -> int:
    """Calcule la durée totale (en jours) d'une liste de codes de modules."""
    with open(CATALOGUE, encoding="utf-8") as f:
        return sum(int(m["duree_jours"]) for m in csv.DictReader(f) if m["code"] in codes)


def politique_catalogue(messages):
    resultats = messages_outils(messages)
    if not resultats:                                   # 1er tour : le « modèle » choisit un outil
        mot = re.search(r"sur (\w+)", derniere_question(messages).lower()).group(1)
        return AIMessage(content="", tool_calls=[appel("chercher_module", mot_cle=mot)])
    if resultats[-1].name == "chercher_module":          # 2e tour : il enchaîne sur un autre outil
        codes = [m["code"] for m in json.loads(resultats[-1].content)]
        return AIMessage(content="", tool_calls=[appel("duree_totale", codes=codes)])
    modules = json.loads(resultats[0].content)            # 3e tour : réponse finale
    titres = ", ".join(f"{m['code']} « {m['titre']} »" for m in modules)
    return AIMessage(content=f"{len(modules)} module(s) : {titres}. Durée totale : {resultats[-1].content} jours.")


agent_catalogue = create_agent(
    model=ModeleSimule(politique=politique_catalogue),
    tools=[chercher_module, duree_totale],
    system_prompt="Tu es l'assistant du catalogue de formations.",
)

resultat = agent_catalogue.invoke({"messages": [{"role": "user", "content": "Quels modules sur airflow ? Durée totale ?"}]})
afficher_conversation(resultat["messages"])
print("\nL'agent est un graphe LangGraph :", type(agent_catalogue).__name__, "| nœuds :", list(agent_catalogue.get_graph().nodes))

 Human: Quels modules sur airflow ? Durée totale ?
    AI: 🔧 chercher_module({"mot_cle": "airflow"})
  Tool[chercher_module]: [{"code": "AIR-101", "titre": "Apache Airflow - fondamentaux", "duree_jours": "2", "niveau": "debutant"}, {"code": "AIR-201", "titre": "Apache Airflow - orchestration avancée", "duree_jours": "3", "niveau": "avance"}]
    AI: 🔧 duree_totale({"codes": ["AIR-101", "AIR-201"]})
  Tool[duree_totale]: 5
    AI: 2 module(s) : AIR-101 « Apache Airflow - fondamentaux », AIR-201 « Apache Airflow - orchestration avancée ». Durée totale : 5 jours.

L'agent est un graphe LangGraph : CompiledStateGraph | nœuds : ['__start__', 'model', 'tools', '__end__']


## 1.3 Middleware : personnaliser la boucle d'agent

Les **middlewares** s'insèrent à des points précis de la boucle (`before_model`, `after_model`, `wrap_model_call`, `wrap_tool_call`…).
LangChain en fournit de nombreux prêts à l'emploi (`SummarizationMiddleware`, `HumanInTheLoopMiddleware`, `ModelCallLimitMiddleware`, `PIIMiddleware`, `ToolRetryMiddleware`…).

In [8]:
from langchain.agents.middleware import before_model, wrap_tool_call, ModelCallLimitMiddleware

journal = []


@before_model
def journaliser_appel_modele(state, runtime):
    journal.append(f"🧠 appel du modèle avec {len(state['messages'])} message(s) dans le contexte")
    return None  # aucune modification de l'état


@wrap_tool_call
def chronometrer_outil(request, handler):
    debut = time.perf_counter()
    resultat = handler(request)
    journal.append(f"🔧 outil {request.tool_call['name']} exécuté en {(time.perf_counter() - debut) * 1000:.2f} ms")
    return resultat


agent_instrumente = create_agent(
    model=ModeleSimule(politique=politique_catalogue),
    tools=[chercher_module, duree_totale],
    middleware=[journaliser_appel_modele, chronometrer_outil, ModelCallLimitMiddleware(run_limit=5)],
)
resultat = agent_instrumente.invoke({"messages": [{"role": "user", "content": "Quels modules sur langgraph ? Durée totale ?"}]})
print("\n".join(journal))
print("\nRéponse :", resultat["messages"][-1].content)

🧠 appel du modèle avec 1 message(s) dans le contexte
🔧 outil chercher_module exécuté en 0.89 ms
🧠 appel du modèle avec 3 message(s) dans le contexte
🔧 outil duree_totale exécuté en 0.84 ms
🧠 appel du modèle avec 5 message(s) dans le contexte

Réponse : 2 module(s) : LGR-101 « LangGraph - agents avec état », LGR-201 « LangGraph - déploiement et observabilité ». Durée totale : 3 jours.


## 1.4 Sortie structurée

Avec `response_format`, l'agent renvoie un objet **Pydantic validé** dans `structured_response`
(stratégie `ToolStrategy` : le modèle « appelle » un outil portant le nom du schéma).

In [9]:
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy


class FicheModule(BaseModel):
    """Fiche descriptive d'un module de formation."""
    code: str = Field(description="Code du module")
    titre: str
    duree_jours: int = Field(ge=1)
    niveau: str


def politique_fiche(messages):
    resultats = messages_outils(messages)
    if not resultats:
        return AIMessage(content="", tool_calls=[appel("chercher_module", mot_cle="kafka")])
    module = json.loads(resultats[-1].content)[0]
    return AIMessage(content="", tool_calls=[appel("FicheModule", **module)])


agent_fiche = create_agent(ModeleSimule(politique=politique_fiche), tools=[chercher_module],
                           response_format=ToolStrategy(FicheModule))
fiche = agent_fiche.invoke({"messages": [{"role": "user", "content": "Donne la fiche du module Kafka"}]})["structured_response"]
print(type(fiche).__name__, "→", fiche)
print("durée (int validé par Pydantic) :", fiche.duree_jours, type(fiche.duree_jours))

FicheModule → code='KAF-101' titre='Apache Kafka - fondamentaux' duree_jours=2 niveau='debutant'
durée (int validé par Pydantic) : 2 <class 'int'>


## 1.5 (optionnel) Le même agent avec un vrai modèle

`init_chat_model("fournisseur:modèle")` instancie n'importe quel modèle supporté (Anthropic, OpenAI, Google, Ollama…).
Pour tester avec un vrai LLM, définissez la clé avant d'exécuter la cellule, par exemple :
`import os; os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."`. Sans clé, la cellule est ignorée.

In [10]:
import os
from langchain.chat_models import init_chat_model

MODELE_REEL = os.environ.get("MODELE_REEL", "anthropic:claude-haiku-4-5")

if os.environ.get("ANTHROPIC_API_KEY"):
    try:
        agent_reel = create_agent(init_chat_model(MODELE_REEL), tools=[chercher_module, duree_totale],
                                  system_prompt="Tu es l'assistant du catalogue de formations. Réponds en français.")
        r = agent_reel.invoke({"messages": [{"role": "user", "content": "Quels modules parlent de Spark ? Durée totale ?"}]})
        afficher_conversation(r["messages"])
    except Exception as exc:
        print("⚠️ Appel au modèle réel impossible :", exc)
else:
    print("ℹ️ Pas de ANTHROPIC_API_KEY : cellule optionnelle ignorée (le reste du handlab n'en a pas besoin).")

ℹ️ Pas de ANTHROPIC_API_KEY : cellule optionnelle ignorée (le reste du handlab n'en a pas besoin).


# Partie 2 — LangGraph : l'environnement d'exécution d'orchestration

> **LangGraph** est l'environnement d'exécution d'orchestration : **exécution durable**, **flux continu** (streaming), **intervention humaine** et **persistance**.

## 2.1 Combiner étapes déterministes et étapes agentiques dans un même graphe

Le graphe ci-dessous masque les données personnelles (étape **déterministe**, auditable), puis **route** la demande
avec une règle métier : les remboursements vont vers un humain, le reste vers un nœud **agentique** (piloté par modèle).

In [11]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END


class EtatDemande(TypedDict, total=False):
    demande: str
    demande_masquee: str
    reponse: str
    traitee_par: str


def masquer_donnees(etat: EtatDemande):                       # déterministe
    return {"demande_masquee": re.sub(r"[\w.]+@[\w.]+", "<email>", etat["demande"])}


def router(etat: EtatDemande) -> Literal["agent_llm", "file_humaine"]:   # déterministe (règle métier)
    return "file_humaine" if "rembours" in etat["demande_masquee"].lower() else "agent_llm"


modele_support = ModeleSimule(politique=lambda msgs: AIMessage(content=f"Réponse générée pour : {msgs[-1].content}"))


def agent_llm(etat: EtatDemande):                              # agentique (piloté par modèle)
    return {"reponse": modele_support.invoke(etat["demande_masquee"]).content, "traitee_par": "LLM"}


def file_humaine(etat: EtatDemande):
    return {"reponse": "Demande transmise au service comptable.", "traitee_par": "humain"}


builder = StateGraph(EtatDemande)
builder.add_node(masquer_donnees)
builder.add_node(agent_llm)
builder.add_node(file_humaine)
builder.add_edge(START, "masquer_donnees")
builder.add_conditional_edges("masquer_donnees", router)
builder.add_edge("agent_llm", END)
builder.add_edge("file_humaine", END)
graphe_support = builder.compile()

print(graphe_support.get_graph().draw_mermaid())
for demande in ["Mon lab Airflow plante, écrire à jean.dupont@exemple.fr",
                "Je souhaite un remboursement, contact : marie@exemple.fr"]:
    r = graphe_support.invoke({"demande": demande})
    print(f"[{r['traitee_par']:>6}] {r['demande_masquee']}  →  {r['reponse']}")

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	masquer_donnees(masquer_donnees)
	agent_llm(agent_llm)
	file_humaine(file_humaine)
	__end__([<p>__end__</p>]):::last
	__start__ --> masquer_donnees;
	masquer_donnees -.-> agent_llm;
	masquer_donnees -.-> file_humaine;
	agent_llm --> __end__;
	file_humaine --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

[   LLM] Mon lab Airflow plante, écrire à <email>  →  Réponse générée pour : Mon lab Airflow plante, écrire à <email>
[humain] Je souhaite un remboursement, contact : <email>  →  Demande transmise au service comptable.


## 2.2 Persistance : threads, checkpoints et reprise après redémarrage

Un **checkpointer** enregistre l'état du graphe après chaque étape (*super-step*), dans un **thread** identifié par `thread_id`.
Avec `SqliteSaver`, l'état survit à la fermeture du programme : on le simule en fermant la connexion puis en recréant le graphe.

In [12]:
import sqlite3
from langgraph.graph import MessagesState
from langgraph.checkpoint.sqlite import SqliteSaver

BASE = LAB / "checkpoints.sqlite"


def repondre_avec_memoire(etat: MessagesState):
    nb_questions = sum(isinstance(m, HumanMessage) for m in etat["messages"])
    return {"messages": [AIMessage(content=f"Question n°{nb_questions} reçue : {etat['messages'][-1].content}")]}


constructeur = StateGraph(MessagesState)
constructeur.add_node(repondre_avec_memoire)
constructeur.add_edge(START, "repondre_avec_memoire")
constructeur.add_edge("repondre_avec_memoire", END)

config = {"configurable": {"thread_id": "stagiaire-42"}}

# --- Session 1 --------------------------------------------------------------
connexion = sqlite3.connect(BASE, check_same_thread=False)
graphe = constructeur.compile(checkpointer=SqliteSaver(connexion))
graphe.invoke({"messages": [HumanMessage("Bonjour")]}, config)
graphe.invoke({"messages": [HumanMessage("Qu'est-ce qu'un checkpoint ?")]}, config)
connexion.close()
print("Session 1 terminée, connexion fermée (≈ redémarrage du programme)\n")

# --- Session 2 : nouveau processus, même base -------------------------------
connexion = sqlite3.connect(BASE, check_same_thread=False)
graphe = constructeur.compile(checkpointer=SqliteSaver(connexion))
r = graphe.invoke({"messages": [HumanMessage("Tu te souviens de moi ?")]}, config)
afficher_conversation(r["messages"])

historique = list(graphe.get_state_history(config))
print(f"\n{len(historique)} checkpoints enregistrés pour le thread 'stagiaire-42' dans {BASE.name}")
print("Un autre thread repart de zéro :",
      graphe.invoke({"messages": [HumanMessage("Salut")]}, {"configurable": {"thread_id": "autre"}})["messages"][-1].content)

Session 1 terminée, connexion fermée (≈ redémarrage du programme)

 Human: Bonjour
    AI: Question n°1 reçue : Bonjour
 Human: Qu'est-ce qu'un checkpoint ?
    AI: Question n°2 reçue : Qu'est-ce qu'un checkpoint ?
 Human: Tu te souviens de moi ?
    AI: Question n°3 reçue : Tu te souviens de moi ?

9 checkpoints enregistrés pour le thread 'stagiaire-42' dans checkpoints.sqlite
Un autre thread repart de zéro : Question n°1 reçue : Salut


## 2.3 Exécution durable : reprendre là où l'on s'était arrêté

Un pipeline en 3 étapes subit une **panne** à l'étape 2. Grâce aux checkpoints, la relance (`invoke(None, config)`)
reprend **à l'étape en échec** : l'étape 1, déjà validée, n'est **pas** ré-exécutée.
Le paramètre `durability="sync"` force l'écriture du checkpoint avant de passer à l'étape suivante.

In [13]:
import operator
from collections import Counter
from typing import Annotated
from langgraph.checkpoint.memory import InMemorySaver

executions = Counter()
panne = {"active": True}


class EtatPipeline(TypedDict):
    journal: Annotated[list[str], operator.add]   # « reducer » : les listes s'additionnent


def extraire(etat):
    executions["extraire"] += 1
    return {"journal": ["1. extraction de 10 000 lignes (coûteuse)"]}


def appeler_api(etat):
    executions["appeler_api"] += 1
    if panne["active"]:
        raise ConnectionError("API partenaire indisponible (panne simulée)")
    return {"journal": ["2. enrichissement via l'API partenaire"]}


def charger(etat):
    executions["charger"] += 1
    return {"journal": ["3. chargement dans l'entrepôt"]}


b = StateGraph(EtatPipeline)
for noeud in (extraire, appeler_api, charger):
    b.add_node(noeud)
b.add_edge(START, "extraire"); b.add_edge("extraire", "appeler_api")
b.add_edge("appeler_api", "charger"); b.add_edge("charger", END)
pipeline = b.compile(checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "etl-2026-09-24"}}

try:
    pipeline.invoke({"journal": []}, cfg, durability="sync")
except ConnectionError as exc:
    print("💥 Panne :", exc)

etat = pipeline.get_state(cfg)
print("Étape en attente de reprise :", etat.next, "| journal sauvegardé :", etat.values["journal"])

panne["active"] = False                     # l'API est rétablie
final = pipeline.invoke(None, cfg, durability="sync")    # None = reprendre le thread
print("\nJournal final :", *final["journal"], sep="\n  ")
print("\nNombre d'exécutions par étape :", dict(executions))
assert executions["extraire"] == 1, "l'extraction ne doit pas être rejouée"

💥 Panne : API partenaire indisponible (panne simulée)
Étape en attente de reprise : ('appeler_api',) | journal sauvegardé : ['1. extraction de 10 000 lignes (coûteuse)']

Journal final :
  1. extraction de 10 000 lignes (coûteuse)
  2. enrichissement via l'API partenaire
  3. chargement dans l'entrepôt

Nombre d'exécutions par étape : {'extraire': 1, 'appeler_api': 2, 'charger': 1}


## 2.4 Flux continu (streaming)

`stream()` diffuse l'exécution au fil de l'eau selon plusieurs **modes** combinables :
`updates` (mise à jour d'état après chaque nœud), `messages` (jetons du LLM un par un), `custom` (données émises par le code via `get_stream_writer`), `values`, `debug`…

In [14]:
from langgraph.config import get_stream_writer
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel

modele_bavard = GenericFakeChatModel(messages=iter([AIMessage(
    content="LangGraph diffuse chaque jeton dès qu'il est produit par le modèle.")]))


def preparer(etat: MessagesState):
    ecrire_flux = get_stream_writer()
    for i in range(1, 4):
        ecrire_flux({"progression": f"préparation {i}/3"})
    return {}


def generer(etat: MessagesState):
    return {"messages": [modele_bavard.invoke(etat["messages"])]}


b = StateGraph(MessagesState)
b.add_node(preparer); b.add_node(generer)
b.add_edge(START, "preparer"); b.add_edge("preparer", "generer"); b.add_edge("generer", END)
graphe_flux = b.compile()

en_cours = False
for mode, donnees in graphe_flux.stream({"messages": [HumanMessage("Explique le streaming")]},
                                         stream_mode=["custom", "updates", "messages"]):
    if mode == "messages":
        morceau, meta = donnees
        if not en_cours:
            print(f"💬 messages (jetons du nœud « {meta['langgraph_node']} ») : ", end="")
            en_cours = True
        print(morceau.content, end="|", flush=True)
        time.sleep(0.05)                    # pour visualiser l'arrivée progressive des jetons
        continue
    if en_cours:
        print(); en_cours = False
    if mode == "custom":
        print("📣 custom   :", donnees)
    else:
        print("🔄 updates  :", {noeud: (None if maj is None else list(maj)) for noeud, maj in donnees.items()})

📣 custom   : {'progression': 'préparation 1/3'}
📣 custom   : {'progression': 'préparation 2/3'}
📣 custom   : {'progression': 'préparation 3/3'}
🔄 updates  : {'preparer': None}
💬 messages (jetons du nœud « generer ») : LangGraph| |diffuse| |chaque| |jeton| |dès| |qu'il| |est| |produit| |par| |le| |modèle.|
🔄 updates  : {'generer': ['messages']}


## 2.5 Intervention humaine (human-in-the-loop)

`interrupt()` met le graphe **en pause** (l'état est sauvegardé par le checkpointer) et renvoie une charge utile à l'application.
L'humain peut **inspecter** (`get_state`) et **modifier** l'état (`update_state`) à tout moment, puis reprendre avec `Command(resume=...)`.

In [15]:
from langgraph.types import interrupt, Command


class EtatPublication(TypedDict, total=False):
    sujet: str
    brouillon: str
    priorite: str
    statut: str


def rediger(etat):
    return {"brouillon": f"Annonce : nouvelle session « {etat['sujet']} » en octobre.", "priorite": "normale"}


def validation_humaine(etat):
    decision = interrupt({"question": "Publier ce brouillon ?", "brouillon": etat["brouillon"]})
    if decision["action"] == "modifier":
        return {"brouillon": decision["texte"], "statut": "modifié puis approuvé"}
    return {"statut": "approuvé" if decision["action"] == "approuver" else "rejeté"}


def publier(etat):
    return {"statut": etat["statut"] + " → publié" if etat["statut"] != "rejeté" else "non publié"}


b = StateGraph(EtatPublication)
b.add_node(rediger); b.add_node(validation_humaine); b.add_node(publier)
b.add_edge(START, "rediger"); b.add_edge("rediger", "validation_humaine")
b.add_edge("validation_humaine", "publier"); b.add_edge("publier", END)
graphe_hitl = b.compile(checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "annonce-1"}}

r = graphe_hitl.invoke({"sujet": "LangGraph avancé"}, cfg)
print("⏸️  Pause :", r["__interrupt__"][0].value)
print("   Prochain nœud :", graphe_hitl.get_state(cfg).next)

# L'humain inspecte puis modifie directement l'état pendant la pause
graphe_hitl.update_state(cfg, {"priorite": "haute"})
print("   État modifié  :", graphe_hitl.get_state(cfg).values["priorite"])

# … puis reprend l'exécution avec sa décision
r = graphe_hitl.invoke(Command(resume={"action": "modifier",
                                        "texte": "📢 Nouvelle session LangGraph avancé le 12 octobre !"}), cfg)
print("▶️  Reprise :", r)

⏸️  Pause : {'question': 'Publier ce brouillon ?', 'brouillon': 'Annonce : nouvelle session « LangGraph avancé » en octobre.'}
   Prochain nœud : ('validation_humaine',)
   État modifié  : haute
▶️  Reprise : {'sujet': 'LangGraph avancé', 'brouillon': '📢 Nouvelle session LangGraph avancé le 12 octobre !', 'priorite': 'haute', 'statut': 'modifié puis approuvé → publié'}


## 2.6 Mémoire à court terme et à long terme

- **Court terme** : les messages d'un *thread* (checkpointer, vu en 2.2).
- **Long terme** : un **store** partagé entre threads/sessions, organisé en *namespaces* (ici par utilisateur).
Le nœud y accède via `runtime.store` ; l'identifiant utilisateur est passé dans le **contexte** d'exécution.

In [16]:
from dataclasses import dataclass
from langgraph.runtime import Runtime
from langgraph.store.memory import InMemoryStore


@dataclass
class Contexte:
    user_id: str


def assistant_memoire(etat: MessagesState, runtime: Runtime[Contexte]):
    espace = ("stagiaires", runtime.context.user_id)
    texte = etat["messages"][-1].content
    if texte.lower().startswith("retiens"):
        runtime.store.put(espace, "preference", {"valeur": texte.split(":", 1)[1].strip()})
        return {"messages": [AIMessage(content="C'est noté pour toutes nos prochaines sessions.")]}
    souvenir = runtime.store.get(espace, "preference")
    info = souvenir.value["valeur"] if souvenir else "aucune préférence connue"
    return {"messages": [AIMessage(content=f"Préférence mémorisée : {info}")]}


b = StateGraph(MessagesState, context_schema=Contexte)
b.add_node(assistant_memoire); b.add_edge(START, "assistant_memoire"); b.add_edge("assistant_memoire", END)
graphe_memoire = b.compile(checkpointer=InMemorySaver(), store=InMemoryStore())

def dire(texte, thread, user):
    r = graphe_memoire.invoke({"messages": [HumanMessage(texte)]},
                              {"configurable": {"thread_id": thread}}, context=Contexte(user_id=user))
    print(f"[{user} / {thread}] {texte!r:<45} → {r['messages'][-1].content}")

dire("Retiens: je préfère les labs en Python", "session-lundi", "alice")
dire("Quelle est ma préférence ?", "session-mardi", "alice")   # autre thread, même utilisatrice
dire("Quelle est ma préférence ?", "session-mardi", "bob")     # autre utilisateur

[alice / session-lundi] 'Retiens: je préfère les labs en Python'      → C'est noté pour toutes nos prochaines sessions.
[alice / session-mardi] 'Quelle est ma préférence ?'                  → Préférence mémorisée : je préfère les labs en Python
[bob / session-mardi] 'Quelle est ma préférence ?'                  → Préférence mémorisée : aucune préférence connue


# Partie 3 — Deep Agents : un agent « tout équipé » construit sur LangGraph

> **Deep Agents** est un ensemble d'agents : **planification**, **sous-agents**, **outils de système de fichiers** et **gestion du contexte** sur LangGraph.

`create_deep_agent()` renvoie un graphe LangGraph compilé déjà équipé de :

| Capacité | Outils / mécanisme |
|---|---|
| Planification | `write_todos` (via `TodoListMiddleware`) |
| Sous-agents | `task` : délègue une sous-tâche à un agent au **contexte isolé** |
| Système de fichiers | `ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`, `delete` sur un *backend* (état, disque, store…) |
| Gestion du contexte | résultats d'outils volumineux déportés dans des fichiers, résumé automatique de l'historique, mémoire `AGENTS.md` |

## 3.1 Un Deep Agent qui planifie, délègue, lit et écrit des fichiers

Le scénario utilise l'espace `lab_ecosysteme/espace_agent/` généré en 0.2 (backend `FilesystemBackend` : les fichiers sont réellement écrits sur le disque).

In [17]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.agents.middleware import TodoListMiddleware

ESPACE = LAB / "espace_agent"
consignes_vues = {}


@tool
def exporter_journal(nb_lignes: int = 30000) -> str:
    """Exporte le journal d'exécution complet des labs (très volumineux)."""
    return "\n".join(f"ligne {i}: lab=OK latence={i % 97} ms" for i in range(nb_lignes))


def plan(statut_1, statut_2, statut_3):
    return [{"content": "Lister les documents", "status": statut_1},
            {"content": "Déléguer l'analyse à un sous-agent", "status": statut_2},
            {"content": "Rédiger le rapport", "status": statut_3}]


def politique_orchestrateur(messages):
    consignes_vues["AGENTS.md chargé"] = "Toujours répondre en français" in str(messages[0].content)
    resultats = messages_outils(messages)
    dernier = resultats[-1].name if resultats else None
    if dernier is None:
        return AIMessage(content="", tool_calls=[appel("write_todos", todos=plan("in_progress", "pending", "pending"))])
    if dernier == "write_todos" and len(resultats) == 1:
        return AIMessage(content="", tool_calls=[appel("ls", path="/docs")])
    if dernier == "ls":
        return AIMessage(content="", tool_calls=[appel("task", subagent_type="analyste",
                         description="Lis les fichiers de /docs et résume chacun en une ligne.")])
    if dernier == "task":
        return AIMessage(content="", tool_calls=[appel("exporter_journal", nb_lignes=30000)])
    if dernier == "exporter_journal":
        synthese = next(m.content for m in resultats if m.name == "task")
        return AIMessage(content="", tool_calls=[appel("write_file", file_path="/rapports/synthese.md",
                         content="# Synthèse de l'écosystème\n\n" + synthese + "\n")])
    if dernier == "write_file":
        return AIMessage(content="", tool_calls=[appel("write_todos", todos=plan("completed", "completed", "completed"))])
    return AIMessage(content="Rapport rédigé dans /rapports/synthese.md ✅")


def politique_analyste(messages):
    resultats = messages_outils(messages)
    if not resultats:
        return AIMessage(content="", tool_calls=[appel("glob", pattern="/docs/*.md")])
    if resultats[-1].name == "glob":
        fichiers = re.findall(r"/docs/[\w.]+\.md", resultats[-1].content)
        return AIMessage(content="", tool_calls=[appel("read_file", file_path=f) for f in fichiers])
    lignes = []
    for m in resultats:
        if m.name == "read_file":
            texte = [l.split("\t", 1)[-1].strip() for l in m.content.splitlines() if l.strip() and not l.startswith("@@")]
            titre = next(l.lstrip("# ").strip() for l in texte if l.startswith("#"))
            lignes.append(f"- **{titre}** : " + " ".join(l for l in texte if not l.startswith("#")))
    return AIMessage(content="\n".join(lignes))


deep_agent = create_deep_agent(
    model=ModeleSimule(politique=politique_orchestrateur),
    tools=[exporter_journal],
    system_prompt="Tu coordonnes la rédaction de rapports de formation.",
    backend=FilesystemBackend(root_dir=ESPACE, virtual_mode=True),
    memory=["/AGENTS.md"],                      # mémoire persistante chargée dans le contexte
    middleware=[TodoListMiddleware()],          # outil de planification write_todos
    subagents=[{
        "name": "analyste",
        "description": "Lit des documents et en produit un résumé concis.",
        "system_prompt": "Tu es un analyste rigoureux.",
        "model": ModeleSimule(politique=politique_analyste),
    }],
)

resultat_deep = deep_agent.invoke({"messages": [{"role": "user", "content": "Fais la synthèse des documents de l'espace de travail."}]})
afficher_conversation(resultat_deep["messages"], largeur=160)

 Human: Fais la synthèse des documents de l'espace de travail.
    AI: 🔧 write_todos({"todos": [{"content": "Lister les documents", "status": "in_progress"}, {"content": "Délé)
  Tool[write_todos]: Updated todo list to [{'content': 'Lister les documents', 'status': 'in_progress'}, {'content': "Déléguer l'analyse à un sous-agent", 'status': 'pending'}, {'co…
    AI: 🔧 ls({"path": "/docs"})
  Tool[ls]: ['/docs/langchain.md', '/docs/langgraph.md', '/docs/langsmith.md']
    AI: 🔧 task({"subagent_type": "analyste", "description": "Lis les fichiers de /docs et résume chacun e)
  Tool[task]: - **LangChain** : Framework d'agents : abstractions et intégrations pour les modèles, les outils et les boucles d'agents. - **LangGraph** : Environnement d'exéc…
    AI: 🔧 exporter_journal({"nb_lignes": 30000})
  Tool[exporter_journal]: Tool result too large, the result of this tool call call_1c9c56b3 was saved in the filesystem at this path: /large_tool_results/call_1c9c56b3  You can read the …
    AI: 🔧

## 3.2 Inspecter ce que le Deep Agent a produit

- le **plan** (`todos`) est conservé dans l'état du graphe ;
- le **sous-agent** a travaillé dans un contexte isolé : seul son résumé est revenu à l'orchestrateur ;
- le **rapport** est un vrai fichier sur le disque ;
- le **journal géant** (≈ 1 Mo) n'a **pas** saturé le contexte : il a été déporté dans `/large_tool_results/` et remplacé par un court message.

In [18]:
print("📋 Plan final :")
for t in resultat_deep["todos"]:
    print(f"   [{t['status']:>9}] {t['content']}")

print("\n🧠 AGENTS.md injecté dans le prompt système :", consignes_vues["AGENTS.md chargé"])

print("\n📄 Contenu de /rapports/synthese.md :")
print((ESPACE / "rapports/synthese.md").read_text(encoding="utf-8"))

msg_journal = next(m for m in resultat_deep["messages"] if isinstance(m, ToolMessage) and m.name == "exporter_journal")
fichier_deporte = next((ESPACE / "large_tool_results").iterdir())
print(f"🗜️ Gestion du contexte : résultat de {fichier_deporte.stat().st_size / 1e6:.1f} Mo déporté dans "
      f"/large_tool_results/{fichier_deporte.name}")
print("   Ce que le modèle voit à la place :", msg_journal.content[:150].replace("\n", " "), "…")

print("\n🗂️ Arborescence de l'espace de travail :")
for f in sorted(ESPACE.rglob("*")):
    if f.is_file():
        print("  ", f.relative_to(ESPACE))

📋 Plan final :
   [completed] Lister les documents
   [completed] Déléguer l'analyse à un sous-agent
   [completed] Rédiger le rapport

🧠 AGENTS.md injecté dans le prompt système : True

📄 Contenu de /rapports/synthese.md :
# Synthèse de l'écosystème

- **LangChain** : Framework d'agents : abstractions et intégrations pour les modèles, les outils et les boucles d'agents.
- **LangGraph** : Environnement d'exécution d'orchestration : exécution durable, flux continu, intervention humaine et persistance.
- **LangSmith** : Plateforme de traçage, d'évaluation, d'invites et de déploiement, indépendante du framework.

🗜️ Gestion du contexte : résultat de 1.0 Mo déporté dans /large_tool_results/call_1c9c56b3
   Ce que le modèle voit à la place : Tool result too large, the result of this tool call call_1c9c56b3 was saved in the filesystem at this path: /large_tool_results/call_1c9c56b3  You can …

🗂️ Arborescence de l'espace de travail :
   AGENTS.md
   docs/langchain.md
   docs/langgraph.md
   

## 3.3 (optionnel) Deep Agent avec un vrai modèle

Avec un vrai LLM, l'agent décide lui-même de son plan, de la délégation et des fichiers à écrire.
Sans `ANTHROPIC_API_KEY`, la cellule est ignorée.

In [19]:
if os.environ.get("ANTHROPIC_API_KEY"):
    try:
        agent_reel = create_deep_agent(model=MODELE_REEL, system_prompt="Réponds en français.",
                                       backend=FilesystemBackend(root_dir=ESPACE, virtual_mode=True),
                                       middleware=[TodoListMiddleware()])
        r = agent_reel.invoke({"messages": [{"role": "user", "content":
            "Lis les fichiers de /docs puis écris un comparatif dans /rapports/comparatif.md"}]})
        afficher_conversation(r["messages"], largeur=160)
    except Exception as exc:
        print("⚠️ Appel au modèle réel impossible :", exc)
else:
    print("ℹ️ Pas de ANTHROPIC_API_KEY : cellule optionnelle ignorée.")

ℹ️ Pas de ANTHROPIC_API_KEY : cellule optionnelle ignorée.


# Partie 4 — LangSmith : traçage, évaluation, invites et déploiement

> **LangSmith** est la plateforme de **traçage**, d'**évaluation**, d'**invites** et de **déploiement** pour différents frameworks (LangChain/LangGraph, mais aussi OpenAI SDK, Anthropic SDK, code Python pur…).

## 4.1 Configuration

Le traçage vers LangSmith s'active par variables d'environnement. **Sans clé**, le handlab fonctionne en mode local :
les traces sont collectées en mémoire avec `TraceurLocal` et l'évaluation tourne sans téléversement.

Pour utiliser le vrai service : `os.environ["LANGSMITH_API_KEY"] = "lsv2_..."` puis ré-exécutez cette cellule.

In [20]:
from langsmith import traceable, tracing_context

PROJET_LANGSMITH = "handlab-ecosysteme-langchain"
LANGSMITH_ACTIF = bool(os.environ.get("LANGSMITH_API_KEY"))
if LANGSMITH_ACTIF:
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_PROJECT"] = PROJET_LANGSMITH

print("Mode LangSmith :", "☁️ en ligne (projet " + PROJET_LANGSMITH + ")" if LANGSMITH_ACTIF
      else "💻 local (aucune donnée envoyée)")

Mode LangSmith : 💻 local (aucune donnée envoyée)


## 4.2 Traçage

Une **trace** est l'arbre des exécutions (*runs*) d'une requête : chaîne, LLM, outils, avec entrées, sorties, durée, jetons et erreurs.
- Tout code LangChain/LangGraph est tracé **automatiquement** ;
- n'importe quelle fonction Python l'est avec le décorateur **`@traceable`** (indépendant du framework).

In [21]:
traceur = TraceurLocal()
agent_catalogue.invoke({"messages": [{"role": "user", "content": "Quels modules sur kafka ? Durée totale ?"}]},
                       config={"callbacks": [traceur], "run_name": "agent_catalogue", "tags": ["handlab"]})
print("Arbre de la trace collectée :\n")
afficher_arbre(traceur.traces[-1])


# @traceable : trace n'importe quelle fonction Python (framework-agnostique)
@traceable(name="nettoyer_question", run_type="tool")
def nettoyer_question(q: str) -> str:
    return re.sub(r"\s+", " ", q).strip()


@traceable(name="pipeline_formation", run_type="chain", tags=["handlab"])
def pipeline_formation(question: str) -> str:
    q = nettoyer_question(question)
    return agent_catalogue.invoke({"messages": [{"role": "user", "content": q}]})["messages"][-1].content


with tracing_context(enabled=LANGSMITH_ACTIF, project_name=PROJET_LANGSMITH):
    print("\n@traceable →", pipeline_formation("Quels   modules sur   spark ? Durée totale ?"))
print("Trace envoyée à LangSmith" if LANGSMITH_ACTIF else "(mode local : @traceable exécute la fonction sans rien envoyer)")

Arbre de la trace collectée :

✓ [chain] agent_catalogue  11.0 ms
   ✓ [chain] model  1.3 ms
      ✓ [llm] ModeleSimule  0.5 ms
   ✓ [chain] tools  2.3 ms
      ✓ [tool] chercher_module  1.3 ms
   ✓ [chain] model  1.3 ms
      ✓ [llm] ModeleSimule  0.6 ms
   ✓ [chain] tools  1.9 ms
      ✓ [tool] duree_totale  0.9 ms
   ✓ [chain] model  1.0 ms
      ✓ [llm] ModeleSimule  0.4 ms

@traceable → 1 module(s) : SPK-201 « Apache Spark - optimisation ». Durée totale : 3 jours.
(mode local : @traceable exécute la fonction sans rien envoyer)


## 4.3 Évaluation

Une **évaluation** exécute une *cible* (l'agent) sur un **jeu de données** et note chaque sortie avec des **évaluateurs**
(code, LLM-juge, humain). La fonction `evaluate()` du SDK s'utilise ici hors ligne (`upload_results=False`) avec le fichier `jeu_evaluation.jsonl`.

In [22]:
import datetime
import pandas as pd
from langsmith.schemas import Example
from langsmith.evaluation import evaluate

lignes = [json.loads(l) for l in (LAB / "data/jeu_evaluation.jsonl").read_text(encoding="utf-8").splitlines()]
maintenant = datetime.datetime.now(datetime.timezone.utc)
jeu_de_donnees = [Example(id=uuid.uuid4(), dataset_id=uuid.uuid4(), created_at=maintenant,
                          inputs={"question": l["question"]},
                          outputs={"duree": l["duree_attendue"], "codes": l["codes_attendus"]}) for l in lignes]


def cible(inputs: dict) -> dict:
    reponse = agent_catalogue.invoke({"messages": [{"role": "user", "content": inputs["question"]}]})["messages"][-1].content
    return {"reponse": reponse}


def duree_correcte(outputs: dict, reference_outputs: dict) -> dict:
    trouvee = re.search(r"Durée totale : (\d+)", outputs["reponse"])
    return {"key": "duree_correcte", "score": int(bool(trouvee) and int(trouvee.group(1)) == reference_outputs["duree"])}


def codes_complets(outputs: dict, reference_outputs: dict) -> dict:
    presents = [c for c in reference_outputs["codes"] if c in outputs["reponse"]]
    return {"key": "codes_complets", "score": len(presents) / len(reference_outputs["codes"])}


import contextlib, io
with contextlib.redirect_stdout(io.StringIO()):        # masque les messages internes du SDK
    resultats_eval = evaluate(cible, data=jeu_de_donnees, evaluators=[duree_correcte, codes_complets],
                              experiment_prefix="catalogue-v1", upload_results=False)
print("Expérience :", resultats_eval.experiment_name)

tableau = pd.DataFrame([{
    "question": r["example"].inputs["question"],
    "réponse": r["run"].outputs["reponse"][:70] + "…",
    **{res.key: res.score for res in r["evaluation_results"]["results"]},
} for r in resultats_eval])
display(tableau)
print("Scores moyens :", tableau[["duree_correcte", "codes_complets"]].mean().to_dict())

0it [00:00, ?it/s]

Expérience : enchanted-wire-83


,question,réponse,duree_correcte,codes_complets
0,Quels modules sur airflow ? Durée totale ?,2 module(s) : AIR-101 « Apache Airflow - fonda...,1,1.0
1,Quels modules sur langgraph ? Durée totale ?,2 module(s) : LGR-101 « LangGraph - agents ave...,1,1.0
2,Quels modules sur kafka ? Durée totale ?,1 module(s) : KAF-101 « Apache Kafka - fondame...,1,1.0
3,Quels modules sur kubernetes ? Durée totale ?,1 module(s) : K8S-101 « Kubernetes - fondament...,1,1.0


Scores moyens : {'duree_correcte': 1.0, 'codes_complets': 1.0}


## 4.4 Invites (prompts) versionnées

LangSmith stocke les invites comme du code : chaque modification crée un **commit** identifié par un hash, que l'on peut
comparer, tester dans le *Playground* et récupérer depuis le code (`client.pull_prompt("nom:commit")`).
La cellule reproduit ce cycle localement ; avec une clé, elle pousse aussi l'invite dans LangSmith.

In [23]:
import hashlib, difflib

registre_invites = {}   # nom -> liste de commits


def pousser_invite(nom, invite: ChatPromptTemplate):
    contenu = "\n".join(f"{m.__class__.__name__.replace('MessagePromptTemplate', '')}: {m.prompt.template}"
                        for m in invite.messages)
    commit = hashlib.sha256(contenu.encode()).hexdigest()[:8]
    registre_invites.setdefault(nom, []).append({"commit": commit, "invite": invite, "contenu": contenu})
    return commit


def tirer_invite(reference):
    nom, _, commit = reference.partition(":")
    versions = registre_invites[nom]
    return next(v["invite"] for v in versions if v["commit"] == commit) if commit else versions[-1]["invite"]


v1 = pousser_invite("formateur-catalogue", ChatPromptTemplate.from_messages([
    ("system", "Tu es formateur."), ("human", "{question}")]))
v2 = pousser_invite("formateur-catalogue", ChatPromptTemplate.from_messages([
    ("system", "Tu es formateur. Réponds en 2 phrases maximum et cite les codes modules."), ("human", "{question}")]))

print("Historique des commits :", [v["commit"] for v in registre_invites["formateur-catalogue"]])
versions = registre_invites["formateur-catalogue"]
print("\n".join(difflib.unified_diff(versions[0]["contenu"].splitlines(), versions[1]["contenu"].splitlines(),
                                     f"commit {v1}", f"commit {v2}", lineterm="")))

for ref in [f"formateur-catalogue:{v1}", "formateur-catalogue"]:
    chaine = tirer_invite(ref) | ModeleSimule(politique=politique_formateur) | StrOutputParser()
    print(f"\n{ref:<28} →", chaine.invoke({"question": "Que couvre LGR-101 ?"}))

if LANGSMITH_ACTIF:
    try:
        from langsmith import Client
        client = Client()
        url = client.push_prompt("formateur-catalogue", object=tirer_invite("formateur-catalogue"))
        print("\n☁️ Invite poussée dans LangSmith :", url)
        print("   Relue depuis LangSmith :", client.pull_prompt("formateur-catalogue").messages[0].prompt.template)
    except Exception as exc:
        print("⚠️ LangSmith Prompts indisponible :", exc)

Historique des commits : ['be12f58f', '95c33de2']
--- commit be12f58f
+++ commit 95c33de2
@@ -1,2 +1,2 @@
-System: Tu es formateur.
+System: Tu es formateur. Réponds en 2 phrases maximum et cite les codes modules.
 Human: {question}

formateur-catalogue:be12f58f → [Tu es formateur.] Réponse simulée à : « Que couvre LGR-101 ? »

formateur-catalogue          → [Tu es formateur. Réponds en 2 phrases maximum et cite les codes modules.] Réponse simulée à : « Que couvre LGR-101 ? »


## 4.5 Déploiement : servir l'agent avec l'Agent Server

Un projet déployable contient un fichier **`langgraph.json`** qui déclare les graphes exposés. Le même projet se déploie :
- en local avec **`langgraph dev`** (serveur en mémoire, idéal pour développer et pour *LangSmith Studio*) ;
- sur **LangSmith Deployment** (cloud, hybride ou auto-hébergé) avec `langgraph deploy` ou depuis l'interface.

Le serveur expose une API standard (assistants, threads, runs, streaming, cron…) utilisable avec le SDK `langgraph_sdk`.

La cellule suivante installe `langgraph-cli[inmem]` dans un **environnement virtuel dédié** (pour ne perturber ni les paquets du notebook, ni ceux préinstallés par Colab),
puis démarre le serveur en arrière-plan (≈ 1 à 2 min la première fois).

In [24]:
import subprocess, socket, urllib.request

PROJET = LAB / "projet_deploiement"
SERVEUR_VENV = Path("serveur_agent_venv").resolve()      # environnement virtuel dédié au serveur
PY_SERVEUR = SERVEUR_VENV / "bin" / "python"
print(PROJET.joinpath("langgraph.json").read_text())

# Environnement sans PYTHONPATH (Colab en définit un qui exposerait ses paquets système au venv)
env_serveur = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "PYTHONHOME")}
env_serveur.update({"PYTHONNOUSERSITE": "1", "LANGSMITH_TRACING": "false",
                    "LANGGRAPH_CLI_NO_ANALYTICS": "1", "PYTHONUNBUFFERED": "1"})

if not (SERVEUR_VENV / ".installation_ok").exists():
    print("Création d'un environnement virtuel isolé et installation de langgraph-cli[inmem]…")
    shutil.rmtree(SERVEUR_VENV, ignore_errors=True)
    # Un vrai venv (et non un simple dossier ajouté au PYTHONPATH) : les paquets système de Colab
    # (ex. protobuf 5.x dans le paquet « google ») ne peuvent pas masquer ceux du serveur (protobuf 6.x).
    creation = subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(SERVEUR_VENV)],
                              capture_output=True, text=True, env=env_serveur)
    if creation.returncode != 0 or not PY_SERVEUR.exists():      # repli si le module venv est incomplet
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "virtualenv"], check=True)
        shutil.rmtree(SERVEUR_VENV, ignore_errors=True)
        subprocess.run([sys.executable, "-m", "virtualenv", "-q", "--no-setuptools", "--no-wheel", str(SERVEUR_VENV)],
                       check=True, env=env_serveur)
    installation = subprocess.run(
        [sys.executable, "-m", "pip", "--python", str(PY_SERVEUR), "install", "-q", "--disable-pip-version-check",
         "langgraph-cli[inmem]==0.4.32", "langgraph==1.2.12", "langgraph-checkpoint==4.2.0",
         "langchain-core==1.6.5", "langgraph-sdk==0.4.5"],
        capture_output=True, text=True, env=env_serveur)
    if installation.returncode != 0:
        print(installation.stdout[-2000:], installation.stderr[-2000:])
        raise RuntimeError("Échec de l'installation de langgraph-cli dans l'environnement isolé")
    (SERVEUR_VENV / ".installation_ok").touch()

if "serveur" in globals() and serveur.poll() is None:     # ré-exécution : on arrête l'ancien serveur
    serveur.terminate(); serveur.wait(timeout=20)

with socket.socket() as s:                                 # port libre
    s.bind(("127.0.0.1", 0))
    PORT = s.getsockname()[1]
URL_SERVEUR = f"http://127.0.0.1:{PORT}"
os.environ["NO_PROXY"] = "127.0.0.1,localhost," + os.environ.get("NO_PROXY", "")

journal_serveur = open(LAB / "serveur.log", "w")
serveur = subprocess.Popen([str(PY_SERVEUR), "-m", "langgraph_cli", "dev", "--no-browser", "--no-reload",
                            "--host", "127.0.0.1", "--port", str(PORT)],
                           cwd=PROJET, env=env_serveur, stdout=journal_serveur, stderr=subprocess.STDOUT)

SERVEUR_OK = False
sans_proxy = urllib.request.build_opener(urllib.request.ProxyHandler({}))
for _ in range(120):
    try:
        SERVEUR_OK = sans_proxy.open(URL_SERVEUR + "/ok", timeout=2).status == 200
        break
    except Exception:
        if serveur.poll() is not None:
            break
        time.sleep(1)

if SERVEUR_OK:
    print(f"✅ Agent Server démarré sur {URL_SERVEUR}  (docs de l'API : {URL_SERVEUR}/docs)")
else:
    print("⚠️ Le serveur n'a pas démarré ; dernières lignes du journal :")
    print("\n".join((LAB / "serveur.log").read_text().splitlines()[-15:]))

{
  "dependencies": [
    "."
  ],
  "graphs": {
    "agent_catalogue": "./agent.py:graph"
  }
}
Création d'un environnement virtuel isolé et installation de langgraph-cli[inmem]…
✅ Agent Server démarré sur http://127.0.0.1:53549  (docs de l'API : http://127.0.0.1:53549/docs)


In [25]:
from langgraph_sdk import get_client

if SERVEUR_OK:
    client_serveur = get_client(url=URL_SERVEUR)

    assistants = await client_serveur.assistants.search()
    print("Assistants exposés :", [a["graph_id"] for a in assistants])

    # Exécution sans état (stateless)
    r = await client_serveur.runs.wait(None, "agent_catalogue",
                                       input={"messages": [{"role": "user", "content": "modules kubernetes"}]})
    print("runs.wait    →", r["messages"][-1]["content"])

    # Exécution avec état (thread) + streaming
    thread = await client_serveur.threads.create()
    async for evenement in client_serveur.runs.stream(thread["thread_id"], "agent_catalogue",
            input={"messages": [{"role": "user", "content": "modules airflow"}]}, stream_mode="updates"):
        if evenement.event == "updates":
            print("runs.stream  →", evenement.data)
    etat = await client_serveur.threads.get_state(thread["thread_id"])
    print("Thread persistant côté serveur :", len(etat["values"]["messages"]), "messages")
else:
    print("ℹ️ Serveur indisponible : cellule ignorée.")

Assistants exposés : ['agent_catalogue']
runs.wait    → Modules trouvés : K8S-101 (3 j)
runs.stream  → {'repondre': {'messages': [{'role': 'ai', 'content': 'Modules trouvés : AIR-101 (2 j), AIR-201 (3 j)'}]}}
Thread persistant côté serveur : 2 messages


# Partie 5 — LangSmith Engine : détecter et corriger les problèmes à partir des traces

> **LangSmith Engine** détecte les problèmes dans les traces de votre agent LangGraph et propose des correctifs. Vous pouvez soumettre une **demande d'extraction** (*pull request*) avec le correctif proposé directement depuis l'onglet **Engine**.

**Fonctionnement du service** (LangSmith Cloud) :
1. un administrateur active Engine sur un **projet de traçage** ; on connecte (recommandé) le **dépôt GitHub** de l'agent ;
2. Engine analyse les traces de production et regroupe les **problèmes récurrents** (*issues*) par catégorie (ex. *Silent tool error*, *Hallucination*) ;
3. pour chaque problème : **preuves** (traces), **diagnostic** de la cause racine et **correctif proposé** ;
4. bouton **Open PR** : ouvre une pull request GitHub avec la modification de code ;
5. **Add offline examples** : transforme les traces fautives en exemples d'évaluation pour **valider** le correctif avant déploiement ;
6. le problème est suivi dans le temps et **réouvert automatiquement** s'il réapparaît.

Engine fonctionne avec les agents Deep Agents, LangChain et LangGraph et est facturé en *LangChain Compute Units*.
Comme il nécessite un compte LangSmith Cloud et un dépôt GitHub, les cellules suivantes **reproduisent localement le même cycle**
(détecter → diagnostiquer → proposer un correctif → valider) sur l'outil défectueux généré en 0.2, à des fins pédagogiques.

## 5.1 Produire des traces « de production » avec un agent défectueux

In [26]:
import importlib
sys.path.insert(0, str(LAB / "engine_demo"))
outils_support = importlib.reload(importlib.import_module("outils_support"))   # version défectueuse, même en ré-exécution

print((LAB / "engine_demo/outils_support.py").read_text())


@tool
def consulter_stock(reference: str) -> str:
    """Renvoie le stock d'une référence produit (ex. SKU-1)."""
    return outils_support.consulter_stock(reference)     # résolu à l'appel : un correctif rechargé est pris en compte


def politique_support(messages):
    resultats = messages_outils(messages)
    if not resultats:
        ref = re.search(r"(sku[\s_-]?\d+)", derniere_question(messages), re.I).group(1)
        return AIMessage(content="", tool_calls=[appel("consulter_stock", reference=ref)])
    sortie = resultats[-1].content
    if sortie.startswith("erreur"):
        # Le « modèle » ne voit pas d'exception : il improvise une réponse (hallucination)
        return AIMessage(content="Bonne nouvelle, ce produit est disponible en stock.")
    donnees = json.loads(sortie)
    return AIMessage(content=f"{donnees['reference']} : {donnees['stock']} unité(s) en stock.")


agent_support = create_agent(ModeleSimule(politique=politique_support), tools=[consulter_stock])

questions_production = ["Stock de SKU-1 ?", "Il reste des sku-3 ?", "Stock du SKU 2 ?", "Stock de SKU-2 ?",
                        "Avez-vous du sku_1 ?", "Stock SKU-3 ?", "Combien de SKU 3 ?", "Stock de sku-02 ?"]
traceur_prod = TraceurLocal()
for q in questions_production:
    r = agent_support.invoke({"messages": [{"role": "user", "content": q}]}, config={"callbacks": [traceur_prod]})
    print(f"{q:<26} → {r['messages'][-1].content}")
print(f"\n{len(traceur_prod.traces)} traces collectées")

import json

STOCK = {"SKU-1": 12, "SKU-2": 0, "SKU-3": 5}


def consulter_stock(reference: str) -> str:
    """Renvoie le stock d'une référence produit au format JSON."""
    try:
        ref = reference.strip().upper()
        return json.dumps({"reference": ref, "stock": STOCK[ref]})
    except Exception as exc:
        # BUG : l'erreur est « avalée » et renvoyée comme un texte ordinaire
        return f"erreur: référence inconnue ({exc})"

Stock de SKU-1 ?           → SKU-1 : 12 unité(s) en stock.
Il reste des sku-3 ?       → SKU-3 : 5 unité(s) en stock.
Stock du SKU 2 ?           → Bonne nouvelle, ce produit est disponible en stock.
Stock de SKU-2 ?           → SKU-2 : 0 unité(s) en stock.
Avez-vous du sku_1 ?       → Bonne nouvelle, ce produit est disponible en stock.
Stock SKU-3 ?              → SKU-3 : 5 unité(s) en stock.
Combien de SKU 3 ?         → Bonne nouvelle, ce produit est disponible en stock.
Stock de sku-02 ?          → Bonne nouvelle, ce produit est disponible en st

## 5.2 Détection et diagnostic des problèmes récurrents

Le « mini-Engine » parcourt les arbres de traces, repère les motifs fautifs et les **regroupe en problèmes** :
- **Silent tool error** : l'outil s'est terminé « avec succès » mais sa sortie contient une erreur ;
- **Hallucination** : la réponse finale affirme un fait que les sorties d'outils ne justifient pas.

In [27]:
from collections import defaultdict


def parcourir(run):
    yield run
    for enfant in run.child_runs:
        yield from parcourir(enfant)


def sortie_texte(run):
    sortie = (run.outputs or {}).get("output")
    return str(getattr(sortie, "content", sortie))


def detecter_problemes(traces):
    problemes = defaultdict(list)
    for trace in traces:
        question = trace.inputs["messages"][0]["content"] if isinstance(trace.inputs["messages"][0], dict) \
            else trace.inputs["messages"][0].content
        reponse = trace.outputs["messages"][-1].content
        for run in parcourir(trace):
            if run.run_type == "tool" and run.error is None and sortie_texte(run).lower().startswith("erreur"):
                preuve = {"trace_id": str(trace.id)[-8:], "question": question,
                          "entree_outil": dict(run.inputs), "sortie_outil": sortie_texte(run)}
                problemes[("Silent tool error", run.name)].append(preuve)
                if "en stock" in reponse and "disponible" in reponse:
                    problemes[("Hallucination", "réponse finale")].append({**preuve, "reponse": reponse})
    return problemes


problemes = detecter_problemes(traceur_prod.traces)
for (categorie, composant), preuves in problemes.items():
    taux = len(preuves) / len(traceur_prod.traces)
    print(f"🚨 {categorie} — {composant} : {len(preuves)} occurrence(s) / {len(traceur_prod.traces)} traces ({taux:.0%})")
    for p in preuves:
        print(f"     trace …{p['trace_id']} | {p['question']!r} → outil : {p['sortie_outil'][:60]}")

# Diagnostic de la cause racine : comparer les entrées fautives aux références valides
entrees_fautives = sorted({p["entree_outil"]["reference"] for p in problemes[("Silent tool error", "consulter_stock")]})
print("\n🔎 Diagnostic")
print("   Références valides :", list(outils_support.STOCK))
print("   Entrées en échec   :", entrees_fautives)
print("   Cause racine       : les références ne sont pas normalisées (espace, '_', zéros de tête) et l'exception")
print("                        est convertie en texte, ce qui pousse le modèle à halluciner une réponse.")

🚨 Silent tool error — consulter_stock : 4 occurrence(s) / 8 traces (50%)
     trace …6a2e760f | 'Stock du SKU 2 ?' → outil : erreur: référence inconnue ('SKU 2')
     trace …0c897bce | 'Avez-vous du sku_1 ?' → outil : erreur: référence inconnue ('SKU_1')
     trace …ffe787f5 | 'Combien de SKU 3 ?' → outil : erreur: référence inconnue ('SKU 3')
     trace …b9c365c7 | 'Stock de sku-02 ?' → outil : erreur: référence inconnue ('SKU-02')
🚨 Hallucination — réponse finale : 4 occurrence(s) / 8 traces (50%)
     trace …6a2e760f | 'Stock du SKU 2 ?' → outil : erreur: référence inconnue ('SKU 2')
     trace …0c897bce | 'Avez-vous du sku_1 ?' → outil : erreur: référence inconnue ('SKU_1')
     trace …ffe787f5 | 'Combien de SKU 3 ?' → outil : erreur: référence inconnue ('SKU 3')
     trace …b9c365c7 | 'Stock de sku-02 ?' → outil : erreur: référence inconnue ('SKU-02')

🔎 Diagnostic
   Références valides : ['SKU-1', 'SKU-2', 'SKU-3']
   Entrées en échec   : ['SKU 2', 'SKU 3', 'sku-02', 'sku_1']
   

## 5.3 Correctif proposé (équivalent du bouton **Open PR**)

Le correctif est produit sous forme de **diff unifié** et d'une description de pull request, écrits dans `engine_demo/`.

In [28]:
chemin_outil = LAB / "engine_demo/outils_support.py"
source_actuelle = chemin_outil.read_text(encoding="utf-8")

source_corrigee = source_actuelle.replace("import json\n", "import json\nimport re\n", 1).replace(
    """        ref = reference.strip().upper()
        return json.dumps({"reference": ref, "stock": STOCK[ref]})
    except Exception as exc:
        # BUG : l'erreur est « avalée » et renvoyée comme un texte ordinaire
        return f"erreur: référence inconnue ({exc})"
""",
    """        numero = re.search(r"(\\d+)", reference)
        ref = f"SKU-{int(numero.group(1))}" if numero else reference.strip().upper()
        return json.dumps({"reference": ref, "stock": STOCK[ref]})
    except KeyError as exc:
        # Une référence réellement inconnue lève une erreur explicite (visible dans la trace)
        raise ValueError(f"Référence inconnue : {reference!r}") from exc
""")
assert source_corrigee != source_actuelle, "le correctif n'a pas pu être appliqué"

diff = "".join(difflib.unified_diff(source_actuelle.splitlines(keepends=True), source_corrigee.splitlines(keepends=True),
                                    "a/engine_demo/outils_support.py", "b/engine_demo/outils_support.py"))
(LAB / "engine_demo/correctif.patch").write_text(diff, encoding="utf-8")
(LAB / "engine_demo/PULL_REQUEST.md").write_text(f"""# fix(outils_support): normaliser les références produit

**Problème détecté** : Silent tool error sur `consulter_stock` ({len(problemes[("Silent tool error", "consulter_stock")])} traces),
entraînant une Hallucination dans la réponse finale.

**Cause racine** : références non normalisées ({", ".join(entrees_fautives)}) et exception convertie en texte.

**Correctif** : extraction du numéro de référence + levée d'une `ValueError` explicite pour les références inconnues.
""", encoding="utf-8")
print(diff)

--- a/engine_demo/outils_support.py
+++ b/engine_demo/outils_support.py
@@ -1,4 +1,5 @@
 import json
+import re
 
 STOCK = {"SKU-1": 12, "SKU-2": 0, "SKU-3": 5}
 
@@ -6,8 +7,9 @@
 def consulter_stock(reference: str) -> str:
     """Renvoie le stock d'une référence produit au format JSON."""
     try:
-        ref = reference.strip().upper()
+        numero = re.search(r"(\d+)", reference)
+        ref = f"SKU-{int(numero.group(1))}" if numero else reference.strip().upper()
         return json.dumps({"reference": ref, "stock": STOCK[ref]})
-    except Exception as exc:
-        # BUG : l'erreur est « avalée » et renvoyée comme un texte ordinaire
-        return f"erreur: référence inconnue ({exc})"
+    except KeyError as exc:
+        # Une référence réellement inconnue lève une erreur explicite (visible dans la trace)
+        raise ValueError(f"Référence inconnue : {reference!r}") from exc



## 5.4 Validation du correctif sur des exemples hors ligne, puis clôture du problème

Les traces fautives deviennent des **exemples d'évaluation** (équivalent de *Add offline examples*). On mesure le taux de réussite
**avant** et **après** correctif, puis on rejoue le trafic de production pour vérifier que le problème ne se reproduit plus.

In [29]:
reponses_attendues = {"SKU-1": "12 unité(s)", "SKU-2": "0 unité(s)", "SKU-3": "5 unité(s)"}
exemples_hors_ligne = []
for p in problemes[("Silent tool error", "consulter_stock")]:
    numero = int(re.search(r"(\d+)", p["question"]).group(1))
    exemples_hors_ligne.append({"question": p["question"], "attendu": reponses_attendues[f"SKU-{numero}"]})


def taux_reussite():
    ok = 0
    for ex in exemples_hors_ligne:
        reponse = agent_support.invoke({"messages": [{"role": "user", "content": ex["question"]}]})["messages"][-1].content
        ok += ex["attendu"] in reponse
    return ok / len(exemples_hors_ligne)


avant = taux_reussite()
chemin_outil.write_text(source_corrigee, encoding="utf-8")       # fusion de la PR
importlib.reload(outils_support)
apres = taux_reussite()
print(f"📊 Exemples hors ligne ({len(exemples_hors_ligne)}) : réussite avant = {avant:.0%}  →  après = {apres:.0%}")

traceur_prod_2 = TraceurLocal()
for q in questions_production:
    agent_support.invoke({"messages": [{"role": "user", "content": q}]}, config={"callbacks": [traceur_prod_2]})
restants = detecter_problemes(traceur_prod_2.traces)
print("✅ Problème clos : aucune récurrence dans le nouveau trafic" if not restants else f"⚠️ Problèmes restants : {list(restants)}")

📊 Exemples hors ligne (4) : réussite avant = 0%  →  après = 100%
✅ Problème clos : aucune récurrence dans le nouveau trafic


# Partie 6 — LangSmith Fleet : des agents sans code

> **LangSmith Fleet** est le générateur d'agents **sans code** pour les modèles, les intégrations et l'automatisation des tâches routinières.

(Anciennement *LangSmith Agent Builder*.) Dans l'interface web de Fleet, on :
- **décrit l'agent en langage naturel** (Fleet génère sa configuration) ou part d'un **modèle** (*template*) ;
- choisit le **modèle** de langage et connecte des **intégrations** (Gmail, Slack, Google Calendar, GitHub, Linear… via connexion de compte sécurisée) ;
- configure des **approbations** : l'agent se met en pause avant les actions sensibles (envoyer un email, publier…) ;
- ajoute des **déclencheurs** (planification, événement) pour automatiser les tâches récurrentes, et une **mémoire** ;
- utilise l'agent dans un **chat**, dans **Slack**, ou l'**appelle depuis une application** (API compatible `langgraph_sdk`).

Sous le capot, un agent Fleet est un **Deep Agent** exécuté sur l'Agent Server. Les cellules suivantes reproduisent ce que Fleet
fait à partir d'une configuration : la spécification `fleet/agent_spec.json` (générée en 0.2) joue le rôle du formulaire sans code.

## 6.1 De la spécification « sans code » à l'agent

In [30]:
spec = json.loads((LAB / "fleet/agent_spec.json").read_text(encoding="utf-8"))
print(json.dumps(spec, ensure_ascii=False, indent=2))


# --- Catalogue d'intégrations disponibles (dans Fleet : connexion de comptes en un clic) ---
@tool
def lire_tickets() -> str:
    """Lit les tickets de support ouverts."""
    with open(LAB / "data/tickets.csv", encoding="utf-8") as f:
        return json.dumps(list(csv.DictReader(f)), ensure_ascii=False)


boite_envoi = []


@tool
def envoyer_email(destinataire: str, sujet: str, corps: str) -> str:
    """Envoie un email (action sensible : soumise à approbation)."""
    boite_envoi.append({"a": destinataire, "sujet": sujet, "corps": corps})
    return f"Email envoyé à {destinataire}"


INTEGRATIONS = {"tickets": [lire_tickets], "email": [envoyer_email]}


def politique_fleet(messages):
    resultats = messages_outils(messages)
    dernier = resultats[-1].name if resultats else None
    if dernier is None:
        return AIMessage(content="", tool_calls=[appel("lire_tickets")])
    if dernier == "lire_tickets":
        ordre = {"haute": 0, "moyenne": 1, "basse": 2}
        tickets = sorted(json.loads(resultats[-1].content), key=lambda t: ordre[t["priorite"]])
        rapport = "# Tri du jour\n" + "\n".join(f"- [{t['priorite']}] {t['id']} {t['client']} : {t['sujet']}" for t in tickets)
        return AIMessage(content="", tool_calls=[appel("write_file", file_path="/rapports/tri_du_jour.md", content=rapport)])
    if dernier == "write_file":
        nb_hautes = sum("[haute]" in l for l in (LAB / "fleet/rapports/tri_du_jour.md").read_text(encoding="utf-8").splitlines())
        return AIMessage(content="", tool_calls=[appel("envoyer_email", destinataire="support@exemple.fr",
                         sujet="Tri des tickets du jour", corps=f"{nb_hautes} ticket(s) prioritaire(s). Rapport : /rapports/tri_du_jour.md")])
    return AIMessage(content=f"Routine terminée : {resultats[-1].content}.")


def construire_agent(spec):
    """Ce que fait Fleet quand on clique sur « Créer » : assembler un Deep Agent à partir de la configuration."""
    outils = [o for nom in spec["integrations"] for o in INTEGRATIONS[nom]]
    return create_deep_agent(
        model=ModeleSimule(politique=politique_fleet),        # dans Fleet : le modèle choisi dans la liste
        tools=outils,
        system_prompt=spec["instructions"],
        memory=[spec["memoire"]],
        interrupt_on=spec["approbations"],                    # approbations humaines
        backend=FilesystemBackend(root_dir=LAB / "fleet", virtual_mode=True),
        checkpointer=InMemorySaver(),                        # nécessaire pour mettre l'agent en pause
        name=spec["nom"],
    )


agent_fleet = construire_agent(spec)
print(f"\n✅ Agent « {agent_fleet.name} » créé avec les outils :", [o.name for o in (lire_tickets, envoyer_email)])

{
  "nom": "Assistant de tri des tickets",
  "instructions": "Chaque matin, lis les tickets, rédige un rapport de tri et préviens l'équipe support par email.",
  "integrations": [
    "tickets",
    "email"
  ],
  "approbations": {
    "envoyer_email": true
  },
  "declencheur": {
    "type": "planification",
    "cron": "0 8 * * 1-5",
    "fuseau": "Europe/Paris"
  },
  "memoire": "/memoires/AGENTS.md"
}

✅ Agent « Assistant de tri des tickets » créé avec les outils : ['lire_tickets', 'envoyer_email']


## 6.2 Déclencheur planifié et approbation humaine

Le déclencheur `0 8 * * 1-5` lance la routine chaque jour ouvré à 8 h. On simule ici **une exécution planifiée** :
l'agent lit les tickets, écrit son rapport puis **se met en pause** avant d'envoyer l'email, en attendant l'approbation
(dans Fleet : une notification « Approuver / Modifier / Rejeter » dans l'interface ou dans Slack).

In [31]:
from zoneinfo import ZoneInfo

# Prochaine exécution du déclencheur (lun-ven à 8 h, heure de Paris)
fuseau = ZoneInfo(spec["declencheur"]["fuseau"])
maintenant = datetime.datetime.now(fuseau)
prochaine = maintenant.replace(hour=8, minute=0, second=0, microsecond=0)
while prochaine <= maintenant or prochaine.weekday() >= 5:
    prochaine = (prochaine + datetime.timedelta(days=1)).replace(hour=8)
JOURS = ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"]
print(f"⏰ Déclencheur « {spec['declencheur']['cron']} » : prochaine exécution le {JOURS[prochaine.weekday()]} {prochaine:%d/%m/%Y à %H:%M}")

print("\n▶️ Exécution planifiée simulée…")
config_fleet = {"configurable": {"thread_id": f"routine-{maintenant:%Y%m%d}"}}
r = agent_fleet.invoke({"messages": [{"role": "user", "content": "Exécute la routine du matin."}]}, config_fleet)

demande = r["__interrupt__"][0].value["action_requests"][0]
print("\n⏸️ Approbation requise pour :", demande["name"])
print("   ", json.dumps(demande["args"], ensure_ascii=False))
print("   Emails envoyés pendant la pause :", len(boite_envoi))

# L'utilisateur clique sur « Approuver »
r = agent_fleet.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config_fleet)
print("\n✅", r["messages"][-1].content)
print("📨 Boîte d'envoi :", boite_envoi)
print("\n📄 Rapport produit :\n" + (LAB / "fleet/rapports/tri_du_jour.md").read_text(encoding="utf-8"))

⏰ Déclencheur « 0 8 * * 1-5 » : prochaine exécution le vendredi 25/09/2026 à 08:00

▶️ Exécution planifiée simulée…

⏸️ Approbation requise pour : envoyer_email
    {"destinataire": "support@exemple.fr", "sujet": "Tri des tickets du jour", "corps": "2 ticket(s) prioritaire(s). Rapport : /rapports/tri_du_jour.md"}
   Emails envoyés pendant la pause : 0

✅ Routine terminée : Email envoyé à support@exemple.fr.
📨 Boîte d'envoi : [{'a': 'support@exemple.fr', 'sujet': 'Tri des tickets du jour', 'corps': '2 ticket(s) prioritaire(s). Rapport : /rapports/tri_du_jour.md'}]

📄 Rapport produit :
# Tri du jour
- [haute] T-001 ACME : Le lab Kubernetes ne démarre pas
- [haute] T-003 Initech : Notebook Airflow en erreur
- [moyenne] T-004 Umbrella : Question sur le planning
- [basse] T-002 Globex : Demande de facture


## 6.3 Appeler un agent Fleet depuis une application

Un agent Fleet s'appelle avec le SDK `langgraph_sdk` (même API que l'Agent Server de la partie 4.5) :
URL de l'espace Fleet, identifiant de l'agent et clé d'accès personnelle, avec l'en-tête `X-Auth-Scheme: langsmith-api-key`.

La fonction ci-dessous est **identique** dans les deux cas. Si les variables `FLEET_API_URL`, `FLEET_AGENT_ID` et `LANGSMITH_API_KEY`
sont définies, elle appelle votre vrai agent Fleet ; sinon, elle est démontrée sur l'Agent Server local démarré en 4.5.

In [32]:
async def appeler_agent(url, agent_id, message, api_key=None):
    en_tetes = {"X-Auth-Scheme": "langsmith-api-key"} if api_key else None
    client = get_client(url=url, api_key=api_key, headers=en_tetes)
    thread = await client.threads.create()
    resultat = await client.runs.wait(thread["thread_id"], agent_id,
                                      input={"messages": [{"role": "user", "content": message}]})
    return resultat["messages"][-1]["content"]


if all(os.environ.get(v) for v in ("FLEET_API_URL", "FLEET_AGENT_ID", "LANGSMITH_API_KEY")):
    print("☁️ Agent Fleet :", await appeler_agent(os.environ["FLEET_API_URL"], os.environ["FLEET_AGENT_ID"],
                                                  "Que peux-tu faire pour moi ?", os.environ["LANGSMITH_API_KEY"]))
elif SERVEUR_OK:
    print("💻 Agent local (même API) :", await appeler_agent(URL_SERVEUR, "agent_catalogue", "modules langgraph"))
else:
    print("ℹ️ Ni agent Fleet configuré, ni serveur local : cellule ignorée.")

💻 Agent local (même API) : Modules trouvés : LGR-101 (2 j), LGR-201 (1 j)


# Récapitulatif

| Brique | Rôle | Ce que vous avez manipulé |
|---|---|---|
| **LangChain** | framework d'agents | `init_chat_model`, `ChatPromptTemplate`, `@tool`, `create_agent`, middlewares, `response_format` |
| **LangGraph** | runtime d'orchestration | `StateGraph`, arêtes conditionnelles, `SqliteSaver`, reprise après panne, `stream()`, `interrupt()`/`Command`, `Store` |
| **Deep Agents** | agent « tout équipé » | `create_deep_agent`, `write_todos`, sous-agent via `task`, `FilesystemBackend`, `AGENTS.md`, déport des gros résultats |
| **LangSmith** | traçage, évaluation, invites, déploiement | arbres de traces, `@traceable`, `evaluate()`, versions d'invites, `langgraph.json` + Agent Server + `langgraph_sdk` |
| **LangSmith Engine** | amélioration continue à partir des traces | détection *Silent tool error* / *Hallucination*, diagnostic, diff de correctif, validation hors ligne |
| **LangSmith Fleet** | agents sans code | spécification → Deep Agent, intégrations, approbation, déclencheur, appel via SDK |

**Pour aller plus loin** : [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview) ·
[LangChain](https://docs.langchain.com/oss/python/langchain/overview) ·
[Deep Agents](https://docs.langchain.com/oss/python/deepagents/overview) ·
[LangSmith Observability](https://docs.langchain.com/langsmith/observability) ·
[LangSmith Deployment](https://docs.langchain.com/langsmith/deployment) ·
[LangSmith Engine](https://docs.langchain.com/langsmith/engine) ·
[LangSmith Fleet](https://docs.langchain.com/langsmith/fleet)

In [33]:
# Nettoyage : arrêt de l'Agent Server local
if "serveur" in globals() and serveur.poll() is None:
    serveur.terminate()
    serveur.wait(timeout=20)
    print("🛑 Agent Server arrêté")
else:
    print("Aucun serveur à arrêter")

🛑 Agent Server arrêté
